# PS3 SHM — retained rainflow power model

Raw stress files → whole-file rainflow moments → fitting-only exponent selection → positive scalar calibration. This is the portable CPU equivalent of the retained GPU scalar recipe; the scalar has an analytic MAPE fit. The printed metric is the task score `max(0, 1 − MAPE)`.

In [ ]:
from pathlib import Path
import hashlib
import os
import numpy as np
import pandas as pd
import rainflow
from sklearn.model_selection import KFold

def find_data():
    override = os.environ.get('PS3_ROOT')
    candidates = []
    if override:
        root = Path(override)
        candidates += [root, root / '02_Datasets' / 'SHM']
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates += [base / 'data', base / 'SHM' / 'data',
                       base / 'NebulaX-Hackathon-ProblemStatement' / 'PS3' / '02_Datasets' / 'SHM']
    for candidate in candidates:
        if (candidate / 'Train').is_dir() and (candidate / 'Train_Labels.csv').is_file():
            return candidate
    raise FileNotFoundError('Set PS3_ROOT or run this notebook from the repository')

DATA = find_data()
EXPONENTS = (2, 3, 4, 5, 6, 8)
print('SHM data:', DATA)

In [ ]:
def rainflow_moments(path):
    raw = pd.read_csv(path, header=None, dtype=np.float64)
    if raw.shape != (581120, 1):
        raise ValueError(f'Unexpected shape: {path.name}: {raw.shape}')
    values = raw.iloc[:, 0].to_numpy()
    if not np.isfinite(values).all():
        raise ValueError(f'Nonfinite SHM input: {path.name}')
    cycles = np.asarray(list(rainflow.extract_cycles(values)), dtype=float)
    if cycles.ndim != 2 or cycles.shape[1] != 5:
        raise ValueError(f'No valid rainflow cycles: {path.name}')
    if not np.isin(cycles[:, 2], [.5, 1.0]).all():
        raise ValueError('Unexpected rainflow cycle convention')
    amplitude, count = cycles[:, 0] / 2.0, cycles[:, 2]
    result = np.asarray([np.sum(count * amplitude ** exponent) for exponent in EXPONENTS])
    if not np.isfinite(result).all() or (result <= 0).any():
        raise ValueError(f'Invalid SHM moments: {path.name}')
    return result

paths = sorted((DATA / 'Train').glob('*.csv'), key=lambda p: int(p.stem.removeprefix('train')))
labels = pd.read_csv(DATA / 'Train_Labels.csv').set_index('filename')
y = np.asarray([float(labels.loc[p.name, 'damage']) for p in paths])
moments = np.asarray([rainflow_moments(p) for p in paths], dtype=float)
hashes = [hashlib.sha256(p.read_bytes()).hexdigest() for p in paths]
assert len(paths) == 64 and np.isfinite(y).all() and (y > 0).all()
print(f'files={len(paths)}, samples per file=581120, exponents={EXPONENTS}')

In [ ]:
def grouped_folds(indices, hashes, n_splits):
    groups = list(dict.fromkeys(hashes[int(i)] for i in indices))
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=17)
    for fit_group_ix, val_group_ix in splitter.split(groups):
        fit_groups = {groups[i] for i in fit_group_ix}
        val_groups = {groups[i] for i in val_group_ix}
        fit = np.asarray([i for i in indices if hashes[int(i)] in fit_groups])
        val = np.asarray([i for i in indices if hashes[int(i)] in val_groups])
        yield fit, val

def weighted_median(values, weights):
    order = np.argsort(values, kind='stable')
    return float(values[order][np.searchsorted(np.cumsum(weights[order]), weights.sum() / 2.0)])

def fit_scale(indices, moment):
    return weighted_median(y[indices] / moment[indices], moment[indices] / y[indices])

def task_metric(truth, prediction):
    mape = float(np.mean(np.abs(truth - prediction) / np.abs(truth)))
    return max(0.0, 1.0 - mape), mape

def select_exponent(fit_indices):
    inner = list(grouped_folds(fit_indices, hashes, 3))
    scores = []
    for exponent_ix, exponent in enumerate(EXPONENTS):
        truth_parts, prediction_parts = [], []
        base = moments[:, exponent_ix]
        for inner_fit, inner_val in inner:
            scale = fit_scale(inner_fit, base)
            truth_parts.append(y[inner_val])
            prediction_parts.append(scale * base[inner_val])
        _, mape = task_metric(np.concatenate(truth_parts), np.concatenate(prediction_parts))
        scores.append((mape, exponent_ix))
    return min(scores)[1]

all_indices = np.arange(len(paths))
outer = list(grouped_folds(all_indices, hashes, 4))
oof = np.full(len(paths), np.nan)
fold_train_scores = []
selected_outer_exponents = []
for fitting, validation in outer:
    exponent_ix = select_exponent(fitting)
    base = moments[:, exponent_ix]
    scale = fit_scale(fitting, base)
    fit_prediction = scale * base[fitting]
    oof[validation] = scale * base[validation]
    fold_train_scores.append(task_metric(y[fitting], fit_prediction)[0])
    selected_outer_exponents.append(EXPONENTS[exponent_ix])

assert np.isfinite(oof).all()
evaluation_score, evaluation_mape = task_metric(y, oof)
final_exponent_ix = select_exponent(all_indices)
final_base = moments[:, final_exponent_ix]
final_scale = fit_scale(all_indices, final_base)
train_prediction = final_scale * final_base
train_score, train_mape = task_metric(y, train_prediction)
print(f'Train score max(0, 1-MAPE) (all-Train fit, optimistic): {train_score:.6f}; MAPE={train_mape:.6%}')
print(f'Evaluation score max(0, 1-MAPE) (4-fold grouped OOF): {evaluation_score:.6f}; MAPE={evaluation_mape:.6%}')
print(f'Final exponent: {EXPONENTS[final_exponent_ix]}; selected outer exponents: {selected_outer_exponents}')
print(f'Fold Train scores: {[round(v, 6) for v in fold_train_scores]}')